In [7]:
# ============================================
# CELDA 1 · Instalar MySQL Server en Colab
# ============================================
!apt-get update -qq
!apt-get install -y mysql-server mysql-client > /dev/null 2>&1

# Arrancar servicio
!service mysql start

# Verificar
!mysql --version
print("✅ MySQL instalado y corriendo")





W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]
mysql  Ver 8.0.46-0ubuntu0.24.04.4 for Linux on x86_64 ((Ubuntu))
✅ MySQL instalado y corriendo


In [8]:
# ============================================
# CELDA EXTRA · Habilitar root con contraseña
# ============================================
!mysql -u root -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root'; FLUSH PRIVILEGES;"
print("✅ Usuario root actualizado con password 'root'")

✅ Usuario root actualizado con password 'root'


In [9]:
# ============================================
# CELDA 2 · Conectar Python con MySQL
# ============================================
!pip install pymysql -q

import pymysql

conexion = pymysql.connect(
    host='127.0.0.1',
    user='root',
    password='root',
    autocommit=True
)
cursor = conexion.cursor()
print("✅ Conectado a MySQL")

✅ Conectado a MySQL


In [10]:
# ============================================
# CELDA 3 · Crear base de datos
# ============================================
cursor.execute("DROP DATABASE IF EXISTS agro_campo")
cursor.execute("""
    CREATE DATABASE agro_campo
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci
""")
cursor.execute("USE agro_campo")
print("✅ Base de datos 'agro_campo' creada")

✅ Base de datos 'agro_campo' creada


In [11]:
# ============================================
# CELDA 4 · Helper para ejecutar SQL y ver errores
# ============================================
def run_sql(sql, mostrar=True):
    try:
        cursor.execute(sql)
        try:
            filas = cursor.fetchall()
            if mostrar:
                for f in filas:
                    print(f)
        except Exception:
            if mostrar:
                print("✅ Ejecutado OK")
    except Exception as e:
        print(f"❌ ERROR: {e}")

In [12]:
# ============================================
# CELDA 5 · Tabla finca
# ============================================
run_sql("""
CREATE TABLE finca (
    id_finca     INT AUTO_INCREMENT PRIMARY KEY,
    nombre       VARCHAR(100) NOT NULL,
    municipio    VARCHAR(80)  NOT NULL,
    departamento VARCHAR(80)  NOT NULL,
    altitud_msnm INT,
    CONSTRAINT chk_altitud CHECK (altitud_msnm BETWEEN 0 AND 4000)
)
""")
print("✅ Tabla 'finca' creada")

✅ Tabla 'finca' creada


In [13]:
# ============================================
# CELDA 6 · Tabla lote con todas las restricciones
# ============================================
run_sql("""
CREATE TABLE lote (
    id_lote       INT AUTO_INCREMENT PRIMARY KEY,
    codigo        VARCHAR(20)  NOT NULL UNIQUE,
    id_finca      INT          NOT NULL,
    cultivo       ENUM('café','maíz','papa','frijol','cacao') NOT NULL,
    area_ha       DECIMAL(6,2) NOT NULL,
    fecha_siembra DATE         NOT NULL,
    estado        VARCHAR(15)  DEFAULT 'activo',
    CONSTRAINT fk_lote_finca FOREIGN KEY (id_finca) REFERENCES finca(id_finca),
    CONSTRAINT chk_area      CHECK (area_ha > 0)
)
""")
print("✅ Tabla 'lote' creada")

✅ Tabla 'lote' creada


In [14]:
# ============================================
# CELDA 7 · Datos de ejemplo
# ============================================
run_sql("""
INSERT INTO finca (nombre, municipio, departamento, altitud_msnm) VALUES
('Finca La Esperanza', 'Facatativá', 'Cundinamarca', 2600),
('Finca El Cafetal',   'Manizales',  'Caldas',        2150),
('Finca Los Andes',    'Tunja',      'Boyacá',        2800)
""", mostrar=False)

run_sql("""
INSERT INTO lote (codigo, id_finca, cultivo, area_ha, fecha_siembra) VALUES
('ESP-L01', 1, 'papa',   2.50, '2025-03-15'),
('ESP-L02', 1, 'maíz',   3.75, '2025-04-10'),
('CAF-L01', 2, 'café',   1.80, '2024-11-20'),
('AND-L01', 3, 'frijol', 4.20, '2025-02-05')
""", mostrar=False)

print("📋 FINCAS:")
run_sql("SELECT * FROM finca")
print("\n📋 LOTES:")
run_sql("SELECT * FROM lote")

📋 FINCAS:
(1, 'Finca La Esperanza', 'Facatativá', 'Cundinamarca', 2600)
(2, 'Finca El Cafetal', 'Manizales', 'Caldas', 2150)
(3, 'Finca Los Andes', 'Tunja', 'Boyacá', 2800)

📋 LOTES:
(1, 'ESP-L01', 1, 'papa', Decimal('2.50'), datetime.date(2025, 3, 15), 'activo')
(2, 'ESP-L02', 1, 'maíz', Decimal('3.75'), datetime.date(2025, 4, 10), 'activo')
(3, 'CAF-L01', 2, 'café', Decimal('1.80'), datetime.date(2024, 11, 20), 'activo')
(4, 'AND-L01', 3, 'frijol', Decimal('4.20'), datetime.date(2025, 2, 5), 'activo')


In [15]:
# ============================================
# CELDA 8 · Prueba CHECK (debe FALLAR)
# ============================================
print("🔴 Prueba 1.4: área negativa\n")
run_sql("""
INSERT INTO lote (codigo, id_finca, cultivo, area_ha, fecha_siembra)
VALUES ('TEST-01', 1, 'papa', -5.00, '2025-05-01')
""")

🔴 Prueba 1.4: área negativa

❌ ERROR: (3819, "Check constraint 'chk_area' is violated.")


In [16]:
# ============================================
# CELDA 9 · Prueba UNIQUE (debe FALLAR)
# ============================================
print("🔴 Prueba 1.5: código duplicado\n")
run_sql("""
INSERT INTO lote (codigo, id_finca, cultivo, area_ha, fecha_siembra)
VALUES ('ESP-L01', 1, 'maíz', 2.00, '2025-05-02')
""")

🔴 Prueba 1.5: código duplicado

❌ ERROR: (1062, "Duplicate entry 'ESP-L01' for key 'lote.codigo'")


In [17]:
# ============================================
# CELDA 10 · Prueba NOT NULL (debe FALLAR)
# ============================================
print("🔴 Prueba 1.6: falta 'cultivo' (NOT NULL)\n")
run_sql("""
INSERT INTO lote (codigo, id_finca, area_ha, fecha_siembra)
VALUES ('TEST-02', 1, 3.00, '2025-05-03')
""")

🔴 Prueba 1.6: falta 'cultivo' (NOT NULL)



In [18]:
# ============================================
# CELDA 11 · RETO 1.a · ALTER TABLE
# ============================================
run_sql("""
ALTER TABLE lote
ADD COLUMN tipo_riego ENUM('goteo','aspersión','gravedad','secano')
DEFAULT 'secano'
""")
print("✅ Columna 'tipo_riego' agregada")

run_sql("DESCRIBE lote")

✅ Columna 'tipo_riego' agregada
('id_lote', 'int', 'NO', 'PRI', None, 'auto_increment')
('codigo', 'varchar(20)', 'NO', 'UNI', None, '')
('id_finca', 'int', 'NO', 'MUL', None, '')
('cultivo', "enum('café','maíz','papa','frijol','cacao')", 'NO', '', None, '')
('area_ha', 'decimal(6,2)', 'NO', '', None, '')
('fecha_siembra', 'date', 'NO', '', None, '')
('estado', 'varchar(15)', 'YES', '', 'activo', '')
('tipo_riego', "enum('goteo','aspersión','gravedad','secano')", 'YES', '', 'secano', '')


In [19]:
# ============================================
# CELDA 12 · RETO 1.b · Tabla operario
# ============================================
run_sql("""
CREATE TABLE operario (
    id_operario           INT AUTO_INCREMENT PRIMARY KEY,
    documento             VARCHAR(20)  NOT NULL UNIQUE,
    nombre_completo       VARCHAR(120) NOT NULL,
    telefono              VARCHAR(20),
    certificado_aplicador BOOLEAN DEFAULT FALSE
)
""")
print("✅ Tabla 'operario' creada")

✅ Tabla 'operario' creada


In [20]:
# ============================================
# CELDA 13 · RETO 1.c · Insertar y consultar
# ============================================
run_sql("""
INSERT INTO operario (documento, nombre_completo, telefono, certificado_aplicador) VALUES
('1012345678', 'Juan Pérez',       '3105557788', TRUE),
('1098765432', 'María Gómez',      '3112223344', FALSE),
('1033445566', 'Carlos Rodríguez', '3204445566', TRUE)
""", mostrar=False)

print("📋 OPERARIOS:")
run_sql("SELECT * FROM operario")

📋 OPERARIOS:
(1, '1012345678', 'Juan Pérez', '3105557788', 1)
(2, '1098765432', 'María Gómez', '3112223344', 0)
(3, '1033445566', 'Carlos Rodríguez', '3204445566', 1)


In [21]:
# ============================================
# CELDA 14 · Verificación final
# ============================================
print("📋 TABLAS EN agro_campo:")
run_sql("SHOW TABLES")

print("\n📋 ESTRUCTURA DE lote:")
run_sql("DESCRIBE lote")

print("\n📋 ESTRUCTURA DE operario:")
run_sql("DESCRIBE operario")

📋 TABLAS EN agro_campo:
('finca',)
('lote',)
('operario',)

📋 ESTRUCTURA DE lote:
('id_lote', 'int', 'NO', 'PRI', None, 'auto_increment')
('codigo', 'varchar(20)', 'NO', 'UNI', None, '')
('id_finca', 'int', 'NO', 'MUL', None, '')
('cultivo', "enum('café','maíz','papa','frijol','cacao')", 'NO', '', None, '')
('area_ha', 'decimal(6,2)', 'NO', '', None, '')
('fecha_siembra', 'date', 'NO', '', None, '')
('estado', 'varchar(15)', 'YES', '', 'activo', '')
('tipo_riego', "enum('goteo','aspersión','gravedad','secano')", 'YES', '', 'secano', '')

📋 ESTRUCTURA DE operario:
('id_operario', 'int', 'NO', 'PRI', None, 'auto_increment')
('documento', 'varchar(20)', 'NO', 'UNI', None, '')
('nombre_completo', 'varchar(120)', 'NO', '', None, '')
('telefono', 'varchar(20)', 'YES', '', None, '')
('certificado_aplicador', 'tinyint(1)', 'YES', '', '0', '')
